# Title

In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, floatmode="fixed", suppress=True)
rng = np.random.default_rng()

Generator(PCG64) at 0x7FA9438FF580

In [4]:
def compute(s):
    print(f"Computing {s}...")
    return f"{s} result"


class MethodType:
    "Emulate PyMethod_Type in Objects/classobject.c"

    def __init__(self, func, obj):
        self.__func__ = func
        self.__self__ = obj

    def __call__(self, *args, **kwargs):
        func = self.__func__
        obj = self.__self__
        return func(obj, *args, **kwargs)


def issubclassable(cls):
    try:

        class _(cls):
            ...

    except:
        return False
    return True


class BaseDecorator:
    def __init__(self, obj):
        if hasattr(obj, "__func__"):
            self.__func__ = obj.__func__
        else:  # was never decorated before.
            self.__func__ = obj

    def __call__(self, *args, **kwargs):
        return self.__func__.__call__(*args ** kwargs)


def Property(func):
    class Wrapped(property):
        @property
        def __func__(self):
            return self.fget

    return Wrapped(func)


def ClassMethod(func):
    if issubclassable(type(func)):

        class Wrapped(type(func)):
            def __get__(self, obj, cls=None):
                if cls is None:
                    cls = type(obj)
                if hasattr(type(self.__func__), "__get__"):
                    return self.__func__.__get__(cls)
                return MethodType(self.__func__, cls)

        return Wrapped(func)

    return classmethod(func)

In [8]:
class A:
    # python 3.9

    @classmethod
    @property
    def clsproperty(cls):
        """A python 3.9 class-property"""
        return compute(f"{cls.__name__}'s clsproperty")

    @classmethod
    def clsmethod(cls):
        """A python 3.9 class-method"""
        return compute(f"{cls.__name__}'s clsmethod")

    @property
    def instproperty(self):
        """A python 3.9 instance-property"""
        return compute(f"{self}'s instproperty")

    # our modified versions

    @ClassMethod
    @Property
    def myclsproperty(cls):
        """A custom class-property"""
        return compute(f"{cls.__name__}'s myclsproperty")

    @ClassMethod
    def myclsmethod(cls):
        """A custom classmethod"""
        return compute(f"{cls.__name__}'s myclsmethod")

    @Property
    def myinstproperty(self):
        """A custom instance-property"""
        return compute(f"{self}'s myinstproperty")


print(A.__dict__)
help(A)

{'__module__': '__main__', 'clsproperty': <classmethod object at 0x7fa9426dd190>, 'clsmethod': <classmethod object at 0x7fa9426dd280>, 'instproperty': <property object at 0x7fa9426e01d0>, 'myclsproperty': <__main__.ClassMethod.<locals>.Wrapped object at 0x7fa942527a60>, 'myclsmethod': <classmethod object at 0x7fa9426dd370>, 'myinstproperty': <__main__.Property.<locals>.Wrapped object at 0x7fa942863160>, '__dict__': <attribute '__dict__' of 'A' objects>, '__weakref__': <attribute '__weakref__' of 'A' objects>, '__doc__': None}
Computing A's clsproperty...
Computing A's clsproperty...
Computing A's clsproperty...
Computing A's myclsproperty...
Computing A's myclsproperty...
Computing A's myclsproperty...
Computing A's clsproperty...
Help on class A in module __main__:

class A(builtins.object)
 |  Class methods defined here:
 |  
 |  clsmethod() from builtins.type
 |      A python 3.9 class-method
 |  
 |  clsproperty = "A's clsproperty result"
 |  myclsmethod() from builtins.type
 |    

In [10]:
A().instproperty

Computing <__main__.A object at 0x7fa942884bb0>'s instproperty...


"<__main__.A object at 0x7fa942884bb0>'s instproperty result"

In [15]:
isinstance(A.__dict__["instance_property"], property)

True

In [4]:
import inspect

In [5]:
inspect.getsource(help)

TypeError: module, class, method, function, traceback, frame, or code object was expected, got _Helper

In [8]:
isinstance(A.__dict__["myclsproperty"], property)

True

In [3]:
class MethodType:
    "Emulate PyMethod_Type in Objects/classobject.c"

    def __init__(self, func, obj):
        self.__func__ = func
        self.__self__ = obj

    def __call__(self, *args, **kwargs):
        func = self.__func__
        obj = self.__self__
        return func(obj, *args, **kwargs)


class ClassMethod:
    "Emulate PyClassMethod_Type() in Objects/funcobject.c"

    def __init__(self, f):
        self.f = f

    def __get__(self, obj, cls=None):
        if cls is None:
            cls = type(obj)
        if hasattr(type(self.f), "__get__"):
            return self.f.__get__(cls)
        return MethodType(self.f, cls)


class Property:
    "Emulate PyProperty_Type() in Objects/descrobject.c"

    def __init__(self, fget=None, fset=None, fdel=None, doc=None):
        self.fget = fget
        self.fset = fset
        self.fdel = fdel
        if doc is None and fget is not None:
            doc = fget.__doc__
        self.__doc__ = doc

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        if self.fget is None:
            raise AttributeError("unreadable attribute")
        return self.fget(obj)

    def __set__(self, obj, value):
        if self.fset is None:
            raise AttributeError("can't set attribute")
        self.fset(obj, value)

    def __delete__(self, obj):
        if self.fdel is None:
            raise AttributeError("can't delete attribute")
        self.fdel(obj)

    def getter(self, fget):
        return type(self)(fget, self.fset, self.fdel, self.__doc__)

    def setter(self, fset):
        return type(self)(self.fget, fset, self.fdel, self.__doc__)

    def deleter(self, fdel):
        return type(self)(self.fget, self.fset, fdel, self.__doc__)

In [4]:
class ClassProperty(property):
    "Emulate PyClassMethod_Type() in Objects/funcobject.c"

    def __init__(self, f, *args, **kwargs):
        f = property(f)
        super().__init__(f, *args, **kwargs)

    def __get__(self, obj, cls=None):
        if cls is None:
            cls = type(obj)
        if hasattr(type(self.fget), "__get__"):
            return self.fget.__get__(cls)
        return MethodType(self.fget, cls)

In [38]:
A.__dict__["prop"].__dir__()

['__get__',
 '__init__',
 '__new__',
 '__func__',
 '__isabstractmethod__',
 '__dict__',
 '__doc__',
 '__repr__',
 '__hash__',
 '__str__',
 '__getattribute__',
 '__setattr__',
 '__delattr__',
 '__lt__',
 '__le__',
 '__eq__',
 '__ne__',
 '__gt__',
 '__ge__',
 '__reduce_ex__',
 '__reduce__',
 '__subclasshook__',
 '__init_subclass__',
 '__format__',
 '__sizeof__',
 '__dir__',
 '__class__']

In [34]:
A.myclsprop.__func__

AttributeError: 'int' object has no attribute '__func__'

In [101]:
A.__dict__["myclsprop"].__dict__["__func__"]

In [53]:
function = type(lambda: None)

In [62]:
class _(function):
    ...

TypeError: type 'function' is not an acceptable base type

In [76]:
A.__dict__["myclsprop"].__dir__()

['__doc__',
 '__module__',
 '__get__',
 '__dict__',
 '__weakref__',
 '__getattribute__',
 '__set__',
 '__delete__',
 '__init__',
 '__new__',
 'getter',
 'setter',
 'deleter',
 'fget',
 'fset',
 'fdel',
 '__isabstractmethod__',
 '__repr__',
 '__hash__',
 '__str__',
 '__setattr__',
 '__delattr__',
 '__lt__',
 '__le__',
 '__eq__',
 '__ne__',
 '__gt__',
 '__ge__',
 '__reduce_ex__',
 '__reduce__',
 '__subclasshook__',
 '__init_subclass__',
 '__format__',
 '__sizeof__',
 '__dir__',
 '__class__']